# Llama2 训练 Demo（教学版）

这个版本按“从小到大”的思路组织：
1) 配置与模块实现
2) 训练 tokenizer
3) 预训练数据与训练
4) SFT 数据与训练
5) 推理采样

面向新手，解释密度为“中等”。

## 0. 运行提示

- 本 notebook 包含“重训练 tokenizer / 训练模型”等重任务。
- 默认 **不自动执行重任务**，通过 `RUN_HEAVY = True` 手动打开。
- 建议先运行到“模型结构”部分，确认能正常导入与构建。

In [ ]:
from pathlib import Path

# 是否运行重任务（下载数据、训练模型）
RUN_HEAVY = False
# 是否运行轻量 smoke test（检查形状与基本调用）
RUN_SMOKE = True

# 项目路径与常用目录
ROOT = Path("/fs-computility-new/UPDZ11_zhanglujia/zhangchuanxi.p/llm-demo")
LLAMA_DIR = ROOT / "LLamA2_demo"
OUTPUT_DIR = ROOT / "output"
SFT_OUTPUT_DIR = LLAMA_DIR / "sft_output"
TOKENIZER_DIR = LLAMA_DIR / "tokenizer"
DATA_DIR = LLAMA_DIR / "data"

## 1. 依赖与导入

集中导入依赖，减少后面重复。

In [ ]:
import os
import json
import math
import time
import random
import warnings
from typing import Generator

# 数值与深度学习基础
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext

# Hugging Face 模型与输出结构
from transformers import PretrainedConfig, PreTrainedModel, AutoTokenizer
from transformers.modeling_outputs import CausalLMOutputWithPast

# Tokenizer 训练组件
from tokenizers import decoders, models, pre_tokenizers, trainers, Tokenizer
from tokenizers.normalizers import NFKC

warnings.filterwarnings("ignore")

## 2. 超参数与配置

先定义一个配置类，集中保存模型超参数。

In [ ]:
class ModelConfig(PretrainedConfig):
    def __init__(
        self,
        dim: int = 768,          # 模型隐藏维度
        n_layers: int = 12,       # Transformer 层数
        n_heads: int = 16,        # 注意力头数
        n_kv_heads: int = 8,      # KV 头数（GQA 用）
        vocab_size: int = 6144,   # 词表大小
        hidden_dim: int = None,   # MLP 中间层维度
        multiple_of: int = 64,    # MLP 维度对齐到该倍数
        norm_eps: float = 1e-5,   # RMSNorm 的 epsilon
        max_seq_len: int = 512,   # 最大序列长度
        dropout: float = 0.0,     # dropout 概率
        flash_attn: bool = True,  # 是否尝试使用 flash attention
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.dim = dim
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.multiple_of = multiple_of
        self.norm_eps = norm_eps
        self.max_seq_len = max_seq_len
        self.dropout = dropout
        self.flash_attn = flash_attn

# 默认配置实例
args = ModelConfig()

## 3. RMSNorm

RMSNorm 是一种轻量归一化，用于稳定训练。输入输出形状不变。

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float):
        super().__init__()
        self.eps = eps
        # 每个通道一个可学习缩放参数
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        # 以均方根归一化，保持数值稳定
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        # 先归一化，再乘缩放参数，输出形状与输入一致
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

if RUN_SMOKE:
    norm = RMSNorm(args.dim, args.norm_eps)
    x = torch.randn(1, 8, args.dim)
    print("RMSNorm output:", norm(x).shape)

## 4. RoPE 与 repeat_kv

- `repeat_kv` 用于 GQA：把键值头复制到查询头数
- RoPE 用旋转方式注入相对位置信息

In [ ]:
def repeat_kv(x, n_rep):
    """
    将 KV 头复制为与 Q 头一致的数量（GQA）。
    x: (batch, seq_len, n_kv_heads, head_dim)
    n_rep: n_heads // n_kv_heads
    """
    if n_rep == 1:
        return x
    return x.repeat_interleave(n_rep, dim=2)


def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    # 预计算 RoPE 复数旋转频率
    assert dim % 2 == 0, f"dim must be even, got {dim}"
    indices = torch.arange(0, dim, 2)[: (dim // 2)].float()
    freqs = 1.0 / (theta ** (indices / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_cis


class RotaryPositionEmbedding:
    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0):
        assert dim % 2 == 0, f"dim must be even, got {dim}"
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base
        # 预先缓存频率矩阵，避免每次前向重复计算
        self.freqs_cis = precompute_freqs_cis(dim, max_seq_len, base)

    def apply_rotary_emb(self, x: torch.Tensor, start_pos: int = 0):
        # x: (batch, seq_len, n_heads, head_dim)
        batch_size, seq_len, n_heads, head_dim = x.shape
        assert head_dim == self.dim
        assert head_dim % 2 == 0

        # 将实数对映射到复数表示
        x_reshaped = x.float().reshape(batch_size, seq_len, n_heads, head_dim // 2, 2)
        x_complex = torch.view_as_complex(x_reshaped)

        # 取对应位置的旋转系数
        freqs_cis = self.freqs_cis.to(device=x.device, dtype=x_complex.dtype)
        freqs_cis = freqs_cis[start_pos:start_pos + seq_len]
        freqs_cis = freqs_cis.view(1, seq_len, 1, head_dim // 2)

        # 复数乘法实现旋转
        x_rotated_complex = x_complex * freqs_cis
        x_rotated = torch.view_as_real(x_rotated_complex)
        x_rotated = x_rotated.reshape(batch_size, seq_len, n_heads, head_dim)
        return x_rotated.type_as(x)

if RUN_SMOKE:
    x = torch.randn(1, 8, args.n_kv_heads, args.dim // args.n_heads)
    print("repeat_kv:", repeat_kv(x, args.n_heads // args.n_kv_heads).shape)
    rope = RotaryPositionEmbedding(dim=args.dim // args.n_heads, max_seq_len=args.max_seq_len)
    x = torch.randn(1, 8, args.n_heads, args.dim // args.n_heads)
    print("rope:", rope.apply_rotary_emb(x).shape)

## 5. Attention

核心是多头注意力 + 可选 flash attention + RoPE。

In [ ]:
class Attention(nn.Module):
    def __init__(self, args: ModelConfig):
        super().__init__()
        # KV 头数（支持 GQA）
        self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        assert args.n_heads % self.n_kv_heads == 0

        # 模型并行参数（这里默认 1）
        model_parallel_size = getattr(args, "model_parallel_size", 1)
        self.n_local_heads = args.n_heads // model_parallel_size
        self.n_local_kv_heads = self.n_kv_heads // model_parallel_size
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = args.dim // args.n_heads

        # Q/K/V 与输出投影
        self.wq = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(args.n_heads * self.head_dim, args.dim, bias=False)

        # dropout
        self.attn_dropout = nn.Dropout(args.dropout)
        self.resid_dropout = nn.Dropout(args.dropout)
        self.dropout = args.dropout

        # flash attention 可用性检查
        self.flash_attn = hasattr(torch.nn.functional, "scaled_dot_product_attention")
        if not self.flash_attn:
            # 手动注意力时需要因果 mask
            mask = torch.full((1, 1, args.max_seq_len, args.max_seq_len), float("-inf"))
            mask = torch.triu(mask, diagonal=1)
            self.register_buffer("mask", mask)

    def forward(self, x):
        # x: (batch, seq_len, dim)
        batch_size, seq_len, _ = x.shape

        # 线性映射得到 Q/K/V
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)
        xq = xq.view(batch_size, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(batch_size, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(batch_size, seq_len, self.n_local_kv_heads, self.head_dim)

        # 应用 RoPE
        rope = RotaryPositionEmbedding(dim=self.head_dim, max_seq_len=seq_len)
        xq = rope.apply_rotary_emb(xq)
        xk = rope.apply_rotary_emb(xk)

        # KV 头复制到与 Q 头数量一致
        xk = repeat_kv(xk, self.n_rep)
        xv = repeat_kv(xv, self.n_rep)

        # 调整为 (batch, n_heads, seq_len, head_dim)
        xq = xq.transpose(1, 2)
        xk = xk.transpose(1, 2)
        xv = xv.transpose(1, 2)

        # 注意力计算
        if self.flash_attn:
            attn_output = torch.nn.functional.scaled_dot_product_attention(
                xq,
                xk,
                xv,
                attn_mask=None if self.dropout == 0.0 else self.mask[:, :, :seq_len, :seq_len],
                dropout_p=self.dropout,
                is_causal=True,
            )
        else:
            # 手动计算注意力分数
            attn_scores = torch.matmul(xq, xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
            attn_scores = attn_scores + self.mask[:, :, :seq_len, :seq_len]
            attn_probs = torch.softmax(attn_scores, dim=-1)
            attn_probs = self.attn_dropout(attn_probs)
            attn_output = torch.matmul(attn_probs, xv)

        # 合并多头并输出
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, self.n_local_heads * self.head_dim)
        output = self.wo(attn_output)
        output = self.resid_dropout(output)
        return output

if RUN_SMOKE:
    attention = Attention(args)
    x = torch.randn(1, 8, args.dim)
    print("Attention output:", attention(x).shape)

## 6. MLP 与 DecoderLayer

Transformer 的“前馈层 + 残差 + 归一化”。

In [ ]:
class MLP(nn.Module):
    def __init__(self, args: ModelConfig):
        super().__init__()
        # 默认隐藏层是 4 倍维度，并对齐到 multiple_of
        hidden_dim = args.hidden_dim or (4 * args.dim)
        if hidden_dim % args.multiple_of != 0:
            hidden_dim = ((hidden_dim + args.multiple_of - 1) // args.multiple_of) * args.multiple_of

        self.fc1 = nn.Linear(args.dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, args.dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(args.dropout)

    def forward(self, x):
        # 两层线性 + GELU + dropout
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, layer_id, args: ModelConfig):
        super().__init__()
        self.attention = Attention(args)
        self.feed_forward = MLP(args)
        self.layer_id = layer_id
        # Pre-Norm 结构
        self.attn_norm = RMSNorm(args.dim, args.norm_eps)
        self.ffn_norm = RMSNorm(args.dim, args.norm_eps)

    def forward(self, x):
        # 自注意力残差
        attn_output = self.attention(self.attn_norm(x))
        h = x + attn_output
        # 前馈残差
        ffn_output = self.feed_forward(self.ffn_norm(h))
        out = h + ffn_output
        return out

if RUN_SMOKE:
    decoder = DecoderLayer(0, args)
    x = torch.randn(1, 8, args.dim)
    print("DecoderLayer output:", decoder(x).shape)

## 7. Llama2 模型

整合 embedding、N 层 decoder、norm 与输出投影。

In [ ]:
class Llama2(PreTrainedModel):
    def __init__(self, args: ModelConfig):
        super().__init__(args)
        self.args = args
        self.vocab_size = args.vocab_size
        self.n_layers = args.n_layers

        # 词嵌入与输出投影
        self.token_embeddings = nn.Embedding(self.vocab_size, args.dim)
        self.dropout = nn.Dropout(args.dropout)
        self.layers = nn.ModuleList([DecoderLayer(i, args) for i in range(self.n_layers)])
        self.norm = RMSNorm(args.dim, args.norm_eps)
        self.output_projection = nn.Linear(args.dim, self.vocab_size, bias=False)
        # 权重共享（词嵌入与输出）
        self.output_projection.weight = self.token_embeddings.weight

        # 预计算 RoPE（注意：实际在 Attention 中使用）
        self.rope = RotaryPositionEmbedding(dim=args.dim // args.n_heads, max_seq_len=args.max_seq_len)
        # 初始化权重
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith("wo.weight"):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * self.n_layers))

        self.last_forward_loss = None
        self.OUT = CausalLMOutputWithPast()
        self._no_split_modules = [name for name, _ in self.named_modules()]

    def _init_weights(self, module):
        # 线性层与嵌入层初始化
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, tokens, targets=None, **kwargs):
        # 兼容 HF 调用参数
        if "input_ids" in kwargs:
            tokens = kwargs["input_ids"]
        if "labels" in kwargs:
            targets = kwargs["labels"]

        # embedding + N 层 decoder + norm
        batch_size, seq_len = tokens.shape
        x = self.token_embeddings(tokens)
        x = self.dropout(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)

        # 训练：输出全部 logits 并计算 loss
        if targets is not None:
            logits = self.output_projection(x)
            loss_fct = nn.CrossEntropyLoss(ignore_index=0, reduction="none")
            self.last_loss = loss_fct(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            # 推理：只取最后一个位置输出
            logits = self.output_projection(x[:, [-1], :])
            self.last_loss = None

        self.OUT.__setitem__("logits", logits)
        self.OUT.__setitem__("last_loss", self.last_loss)
        return self.OUT

    @torch.inference_mode()
    def generate(self, idx, stop_id=None, max_new_tokens=256, temperature=1.0, top_k=None):
        # 逐步自回归生成（无 KV cache）
        index = idx.shape[1]
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.args.max_seq_len else idx[:, -self.args.max_seq_len :]
            logits = self(idx_cond).logits
            logits = logits[:, -1, :]

            # 采样策略
            if temperature == 0.0:
                _, idx_next = torch.topk(logits, k=1, dim=-1)
            else:
                logits = logits / temperature
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float("Inf")
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)

            if idx_next == stop_id:
                break
            idx = torch.cat((idx, idx_next), dim=1)

        # 只返回新增 token
        return idx[:, index:]

if RUN_SMOKE:
    model = Llama2(args)
    tokens = torch.randint(0, args.vocab_size, (1, 8))
    out = model(tokens)
    print("Llama2 logits:", out.logits.shape)

## 8. Tokenizer 训练（可选重任务）

训练 BPE 分词器，数据量大时请谨慎执行。

In [ ]:
def read_texts_from_jsonl(file_path):
    # 逐行读取 JSONL，仅抽取 text 字段
    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            try:
                data = json.loads(line)
                if "text" not in data:
                    raise KeyError(f"Missing 'text' field in line {line_num}")
                yield data["text"]
            except json.JSONDecodeError:
                print(f"Error decoding JSON in line {line_num}")
            except KeyError as e:
                print(e)


def create_tokenizer_config(save_dir):
    # 生成 tokenizer_config.json 与 special_tokens_map.json
    config = {
        "add_bos_token": False,
        "add_eos_token": False,
        "add_prefix_space": False,
        "bos_token": "<|im_start|>",
        "eos_token": "<|im_end|>",
        "pad_token": "<|im_end|>",
        "unk_token": "<unk>",
        "model_max_length": 1000000000000000019884624838656,
        "clean_up_tokenization_spaces": False,
        "tokenizer_class": "PreTrainedTokenizerFast",
        "chat_template": (
            "{% for message in messages %}"
            "{% if message['role'] == 'system' %}"
            "<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
            "{% elif message['role'] == 'user' %}"
            "<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
            "{% elif message['role'] == 'assistant' %}"
            "<|im_start|>assistant\n{{ message['content'] }}<|im_end|>\n"
            "{% endif %}"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
            "{{ '<|im_start|>assistant\n' }}"
            "{% endif %}"
        ),
    }

    os.makedirs(save_dir, exist_ok=True)
    with open(Path(save_dir) / "tokenizer_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=4)

    special_tokens_map = {
        "bos_token": "<|im_start|>",
        "eos_token": "<|im_end|>",
        "unk_token": "<unk>",
        "pad_token": "<|im_end|>",
        "additional_special_tokens": ["<s>", "</s>"],
    }
    with open(Path(save_dir) / "special_tokens_map.json", "w", encoding="utf-8") as f:
        json.dump(special_tokens_map, f, ensure_ascii=False, indent=4)


def train_tokenizer(data_path, save_dir, vocab_size=8192) -> None:
    # 训练 BPE tokenizer
    os.makedirs(save_dir, exist_ok=True)

    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer = NFKC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()

    special_tokens = ["<unk>", "<s>", "</s>", "<|im_start|>", "<|im_end|>"]
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        min_frequency=2,
        show_progress=True,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )

    print(f"Training tokenizer with data from {data_path}")
    texts = read_texts_from_jsonl(data_path)
    tokenizer.train_from_iterator(texts, trainer=trainer, length=os.path.getsize(data_path))

    tokenizer.save(Path(save_dir) / "tokenizer.json")
    create_tokenizer_config(save_dir)
    print(f"Tokenizer saved to {save_dir}")

In [ ]:
if RUN_HEAVY:
    # 训练 tokenizer（数据文件需提前准备）
    data_path = DATA_DIR / "mobvoi_seq_monkey_general_open_corpus.jsonl"
    save_dir = TOKENIZER_DIR
    train_tokenizer(str(data_path), str(save_dir), vocab_size=args.vocab_size)
else:
    print("Skip tokenizer training (RUN_HEAVY=False)")

## 9. 预训练数据集

把 JSONL 每一行转成 (X, Y, loss_mask)。

In [ ]:
class PretrainDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_seq_len=512):
        super().__init__()
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.padding = tokenizer.pad_token_id if tokenizer.pad_token is not None else 0

        # 记录每行的字节偏移，便于随机访问大文件
        self._offsets = []
        with open(data_path, "rb") as f:
            self._offsets.append(0)
            while f.readline():
                self._offsets.append(f.tell())
        self._total_lines = len(self._offsets) - 1

    def __len__(self):
        return self._total_lines

    def __getitem__(self, idx):
        # 读取第 idx 行 JSON
        with open(self.data_path, "rb") as f:
            f.seek(self._offsets[idx])
            line = f.readline().decode("utf-8")
        sample = json.loads(line)

        # 加 BOS token 并截断
        text = f"{self.tokenizer.bos_token}{sample['text']}"
        input_id = self.tokenizer(text).data["input_ids"][: self.max_seq_len]

        # padding + loss mask
        text_len = len(input_id)
        padding_len = self.max_seq_len - text_len
        input_id = input_id + [self.padding] * padding_len
        loss_mask = [1] * text_len + [0] * padding_len

        # 构造 (X, Y, loss_mask)
        input_id = np.array(input_id, dtype=np.int64)
        X = np.array(input_id[:-1], dtype=np.int64)
        Y = np.array(input_id[1:], dtype=np.int64)
        loss_mask = np.array(loss_mask[1:], dtype=np.int64)
        return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(loss_mask)

## 10. 预训练循环

包含学习率调度、梯度累积、保存检查点。

In [ ]:
def Logger(msg):
    # 简单日志输出
    print(msg)


def get_lr(it, all_iters, base_lr, warmup_iters):
    # 余弦退火 + warmup
    min_lr = base_lr / 10
    if it < warmup_iters:
        return base_lr * it / warmup_iters
    if it > all_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (all_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (base_lr - min_lr)


def train_epoch(epoch, model, train_loader, optimizer, scaler, ctx, args):
    # 一个 epoch 的训练循环
    start_time = time.time()
    iters_per_epoch = len(train_loader)
    for step, (X, Y, loss_mask) in enumerate(train_loader):
        # 数据上设备
        X = X.to(args.device)
        Y = Y.to(args.device)
        loss_mask = loss_mask.to(args.device)

        # 动态学习率
        lr = get_lr(epoch * iters_per_epoch + step, args.epochs * iters_per_epoch, args.learning_rate, args.warmup_iters)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        # 前向与损失
        with ctx:
            out = model(X, Y)
            loss = out.last_loss / args.accumulation_steps
            loss_mask = loss_mask.view(-1)
            loss = torch.sum(loss * loss_mask) / loss_mask.sum()

        # 反向传播（混合精度）
        scaler.scale(loss).backward()

        # 梯度累积与优化器 step
        if (step + 1) % args.accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # 日志输出
        if step % args.log_interval == 0:
            spend_time = time.time() - start_time
            Logger(
                "Epoch:[{}/{}]({}/{}) loss:{:.3f} lr:{:.7f} epoch_Time:{}min".format(
                    epoch + 1,
                    args.epochs,
                    step,
                    iters_per_epoch,
                    loss.item() * args.accumulation_steps,
                    optimizer.param_groups[-1]["lr"],
                    spend_time / (step + 1) * iters_per_epoch // 60 - spend_time // 60,
                )
            )

        # 保存模型（周期性）
        if (step + 1) % args.save_interval == 0:
            model.eval()
            ckp = f"{args.save_dir}/pretrain_{args.dim}_{args.n_layers}_{args.vocab_size}.pth"
            state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
            torch.save(state_dict, ckp)
            model.train()

        # 大步长额外保存
        if (step + 1) % 20000 == 0:
            model.eval()
            ckp = f"{args.save_dir}/pretrain_{args.dim}_{args.n_layers}_{args.vocab_size}_step{step+1}.pth"
            state_dict = model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict()
            torch.save(state_dict, ckp)
            model.train()


def init_model_and_tokenizer(args):
    # 加载 tokenizer + 初始化模型
    tokenizer = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR))
    model = Llama2(args)

    # 多卡包装
    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        Logger(f"Using {num_gpus} GPUs with DataParallel")
        model = torch.nn.DataParallel(model)

    model = model.to(args.device)
    Logger(f"LLM params: {sum(p.numel() for p in model.parameters()) / 1e6:.3f} M")
    return model, tokenizer

In [ ]:
if RUN_HEAVY:
    # 训练配置
    if not hasattr(args, "device"):
        args.device = "cuda" if torch.cuda.is_available() else "cpu"
    args.train_data_path = str(DATA_DIR / "mobvoi_seq_monkey_general_open_corpus.jsonl")
    args.batch_size = 8
    args.num_workers = 2
    args.epochs = 1
    args.learning_rate = 3e-4
    args.warmup_iters = 100
    args.accumulation_steps = 4
    args.grad_clip = 1.0
    args.log_interval = 50
    args.save_interval = 2000
    args.save_dir = str(OUTPUT_DIR)

    # 初始化模型与数据
    model, tokenizer = init_model_and_tokenizer(args)
    train_dataset = PretrainDataset(args.train_data_path, tokenizer, max_seq_len=args.max_seq_len)
    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=True,
    )

    # 优化器与混合精度
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
    use_amp = args.device.startswith("cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else nullcontext()

    # 训练
    for epoch in range(args.epochs):
        train_epoch(epoch, model, train_loader, optimizer, scaler, ctx, args)

    Logger("预训练结束")
else:
    print("Skip pretrain (RUN_HEAVY=False)")

## 11. SFT 数据集

把多轮对话 JSON 转成模型可训练的输入。

In [ ]:
from tqdm import tqdm


def convert_message(data):
    # 将原始对话转换为 chat_template 所需格式
    message = [{"role": "system", "content": "你是一个AI助手"}]
    for item in data:
        if item["from"] == "human":
            message.append({"role": "user", "content": item["value"]})
        elif item["from"] == "assistant":
            message.append({"role": "assistant", "content": item["value"]})
    return message


class SFTDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_seq_len=512):
        super().__init__()
        self.data_path = data_path
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.padding = tokenizer.pad_token_id if tokenizer.pad_token is not None else 0

        # 记录每行偏移，避免整文件加载
        self._offsets = []
        with open(data_path, "rb") as f:
            self._offsets.append(0)
            while f.readline():
                self._offsets.append(f.tell())
        self._total_lines = len(self._offsets) - 1

    def __len__(self):
        return self._total_lines

    def generate_loss_mask(self, input_id):
        # 只对 assistant 回复部分计算 loss
        mask = [0] * len(input_id)
        a_sequence = self.tokenizer("<|im_start|>assistant\n")["input_ids"]
        a_len = len(a_sequence)
        n = len(input_id)
        i = 0
        while i <= n - a_len:
            match = True
            for k in range(a_len):
                if input_id[i + k] != a_sequence[k]:
                    match = False
                    break
            if match:
                j = None
                for idx in range(i + a_len, n):
                    if input_id[idx] == self.tokenizer.eos_token_id:
                        j = idx
                        break
                if j is not None:
                    start = i + a_len
                    end = j
                    for pos in range(start, end + 1):
                        if pos < len(mask):
                            mask[pos] = 1
                i += a_len
            else:
                i += 1
        return mask

    def __getitem__(self, idx):
        # 读取第 idx 行对话
        with open(self.data_path, "rb") as f:
            f.seek(self._offsets[idx])
            line = f.readline().decode("utf-8")
        sample = json.loads(line)

        # 拼接成 chat template 并编码
        text = self.tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=False)
        input_id = self.tokenizer(text).data["input_ids"][: self.max_seq_len]
        text_len = len(input_id)

        # padding + loss mask
        padding_len = self.max_seq_len - text_len
        input_id = input_id + [self.padding] * padding_len
        loss_mask = self.generate_loss_mask(input_id)

        # 构造 (X, Y, loss_mask)
        input_id = np.array(input_id)
        X = np.array(input_id[:-1]).astype(np.int64)
        Y = np.array(input_id[1:]).astype(np.int64)
        loss_mask = np.array(loss_mask[1:]).astype(np.int64)
        return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(loss_mask)

## 12. SFT 训练循环

加载预训练权重并继续训练。

In [ ]:
def init_model_for_sft(args):
    # 初始化模型并加载预训练权重
    tokenizer = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR))
    model = Llama2(args)

    ckp = OUTPUT_DIR / "pretrain_768_12_6144.pth"
    state_dict = torch.load(ckp, map_location=args.device)
    unwanted_prefix = "_orig_mod."
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict, strict=False)

    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        Logger(f"Using {num_gpus} GPUs with DataParallel")
        model = torch.nn.DataParallel(model)

    model = model.to(args.device)
    Logger(f"LLM params: {sum(p.numel() for p in model.parameters()) / 1e6:.3f} M")
    return model, tokenizer


if RUN_HEAVY:
    # 训练配置
    if not hasattr(args, "device"):
        args.device = "cuda" if torch.cuda.is_available() else "cpu"
    args.train_data_path = str(DATA_DIR / "BelleGroup_sft.jsonl")
    args.batch_size = 32
    args.num_workers = 8
    args.epochs = 1
    args.learning_rate = 2e-4
    args.warmup_iters = 100
    args.accumulation_steps = 8
    args.grad_clip = 1.0
    args.log_interval = 100
    args.save_interval = 2000
    args.save_dir = str(SFT_OUTPUT_DIR)
    os.makedirs(args.save_dir, exist_ok=True)

    # 初始化模型与数据
    model, tokenizer = init_model_for_sft(args)
    train_dataset = SFTDataset(args.train_data_path, tokenizer, max_seq_len=args.max_seq_len)
    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=True,
    )

    # 优化器与混合精度
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
    use_amp = args.device.startswith("cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else nullcontext()

    # 训练
    for epoch in range(args.epochs):
        train_epoch(epoch, model, train_loader, optimizer, scaler, ctx, args)

    Logger("SFT 训练结束")
else:
    print("Skip SFT (RUN_HEAVY=False)")

## 13. 推理与采样

加载训练好的模型，给定提示词生成文本。

In [ ]:
class TextGenerator:
    def __init__(
        self,
        checkpoint=str(OUTPUT_DIR / "pretrain_768_12_6144.pth"),
        tokenizer_model_path=str(TOKENIZER_DIR),
        seed=42,
        device=None,
        dtype="bfloat16",
    ):
        # 路径与设备
        self.checkpoint = checkpoint
        self.tokenizer_model_path = tokenizer_model_path
        self.seed = seed
        self.device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        self.dtype = dtype
        self.device_type = "cuda" if "cuda" in self.device else "cpu"

        # 随机种子与 TF32
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

        # 自动混合精度上下文
        ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[self.dtype]
        self.ctx = nullcontext() if self.device_type == "cpu" else torch.amp.autocast(device_type=self.device_type, dtype=ptdtype)

        # 加载模型权重
        checkpoint_dict = torch.load(self.checkpoint, map_location=self.device)
        self.model = Llama2(ModelConfig(dim=768, n_layers=12))
        unwanted_prefix = "_orig_mod."
        for k, v in list(checkpoint_dict.items()):
            if k.startswith(unwanted_prefix):
                checkpoint_dict[k[len(unwanted_prefix):]] = checkpoint_dict.pop(k)
        self.model.load_state_dict(checkpoint_dict, strict=False)

        num_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"Model has {num_params / 1e6:.3f} M parameters.")
        self.model.eval()
        self.model.to(self.device)
        self.tokenizer = AutoTokenizer.from_pretrained(self.tokenizer_model_path)

    def chat_template(self, prompt):
        # 对话模板（system + user）
        message = [
            {"role": "system", "content": "你是一个AI助手，你的名字叫小明。"},
            {"role": "user", "content": prompt},
        ]
        return self.tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

    def sft_sample(self, start="Hello!", num_samples=3, max_new_tokens=256, temperature=0.7, top_k=300):
        # SFT 风格采样（带 chat template）
        start = self.chat_template(start)
        start_ids = self.tokenizer(start).data["input_ids"]
        x = torch.tensor(start_ids, dtype=torch.long, device=self.device)[None, ...]
        generated_texts = []
        with torch.no_grad():
            with self.ctx:
                for _ in range(num_samples):
                    y = self.model.generate(x, self.tokenizer.eos_token_id, max_new_tokens, temperature=temperature, top_k=top_k)
                    generated_texts.append(self.tokenizer.decode(y[0].tolist()))
        return generated_texts

    def pretrain_sample(self, start="Hello!", num_samples=3, max_new_tokens=256, temperature=0.7, top_k=300):
        # 预训练风格采样（直接文本）
        if start.startswith("FILE:"):
            with open(start[5:], "r", encoding="utf-8") as f:
                start = f.read()
        start_ids = self.tokenizer(start).data["input_ids"]
        x = torch.tensor(start_ids, dtype=torch.long, device=self.device)[None, ...]
        generated_texts = []
        with torch.no_grad():
            with self.ctx:
                for _ in range(num_samples):
                    y = self.model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
                    generated_texts.append(self.tokenizer.decode(y[0].tolist()))
        return generated_texts

if RUN_SMOKE:
    generator = TextGenerator()
    samples = generator.pretrain_sample(start="<|im_start|>北京大学是", num_samples=1, max_new_tokens=60, temperature=0.8)
    print("Sample:", samples[0])